# 02. Функции, исключения, файлы, классы, генераторы

Неделя 2.

**Как работать с блокнотом**
- Сначала решаешь сам, без ИИ. В задачах «предскажи» сначала пишешь ответ в комментарий, потом запускаешь.
- Непонятное поведение — прогони фрагмент в [pythontutor.com](https://pythontutor.com/) и посмотри на стрелки «имя → объект».
- Ячейки Jupyter можно запускать в любом порядке, и состояние (переменные) копится между ними. Это источник «у меня работало». После всех выполненных заданий: **Kernel → Restart & Run All** — блокнот должен пройти сверху вниз без ошибок.

## Функция проверки `check`

Это **не чёрный ящик** — её тоже нужно понимать построчно.

- `cases` — список пар `(аргументы, ожидаемый результат)`. Аргументы — кортеж, поэтому для одного аргумента пишется `(x,)` с запятой: `(x)` — это просто `x` в скобках.
- `fn(*args)` распаковывает кортеж в позиционные аргументы: `fn(*(1, 2))` то же, что `fn(1, 2)`.
- `_same` сравнивает результаты. Вещественные числа сравниваются **приближённо** (`math.isclose`): `0.1 + 0.2 == 0.3` в Python — `False`, об этом ниже. Списки и кортежи сравниваются поэлементно и **с учётом типа**: если ожидается кортеж, список не подойдёт.
- `assert условие, сообщение` бросает `AssertionError` с сообщением, если условие ложно.
- `!r` в f-строке выводит `repr` значения: строки будут в кавычках, и `'1'` не спутать с `1`.

Дополнительно создаём временную папку для файлов: `tempfile.mkdtemp()` создаёт уникальную папку в системном каталоге временных файлов и возвращает её путь.

In [ ]:
import math

def _same(a, b):
    if isinstance(a, float) or isinstance(b, float):
        if a is None or b is None:
            return a is b
        return math.isclose(a, b, rel_tol=1e-9, abs_tol=1e-12)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return type(a) is type(b) and len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b

def check(fn, cases):
    """cases: список пар (кортеж аргументов, ожидаемый результат)."""
    for args, expected in cases:
        got = fn(*args)
        assert _same(got, expected), f"{fn.__name__}{args!r}: ожидалось {expected!r}, получено {got!r}"
    print(f"{fn.__name__}: OK ({len(cases)} тестов)")

import os, tempfile, inspect
WORKDIR = tempfile.mkdtemp()
print(WORKDIR)

## 1. Тестовые данные

Файл измерений, как из прибора: строка-комментарий, заголовок, корректные строки, битые строки.

**Тонкость с кодировкой.** Всегда указывай `encoding="utf-8"` при открытии текстовых файлов. Без этого Python берёт кодировку системы: на Linux это обычно UTF-8, а на Windows — часто `cp1251`. Файл с кириллицей, записанный на одной машине, на другой прочитается как мусор или упадёт с `UnicodeDecodeError`.

In [ ]:
RAW = """# прибор: spectrometer-2
sample,temp,value
A1,20.5,0.113
A2,21.0,0.127
A3,bad,0.131
# калибровка
A4,21.4,
A5,22.0,0.140
"""
PATH = os.path.join(WORKDIR, "measurements.csv")
with open(PATH, "w", encoding="utf-8") as f:
    f.write(RAW)

with open(PATH, encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        print(i, repr(line))   # repr показывает невидимый символ конца строки \n

## 2. Исключения

Исключение — это объект, который «всплывает» по стеку вызовов, пока его не поймают. Если не поймали, программа завершается и печатает traceback (читать его нужно **снизу вверх**: последняя строка — тип и текст ошибки, выше — где она произошла).

Полная форма:
```python
try:
    ...              # код, который может бросить исключение
except ValueError as e:
    ...              # выполняется, если брошен ValueError (или его подкласс)
else:
    ...              # выполняется, если исключения НЕ было
finally:
    ...              # выполняется ВСЕГДА: и при успехе, и при ошибке
```

Исключения образуют иерархию классов: `except OSError` поймает и `FileNotFoundError`, и `PermissionError`, потому что это его подклассы.

**Правила:**
- ловить **конкретные** исключения, которые ты ожидаешь и умеешь обработать;
- `except Exception:` ловит почти всё, включая опечатки (`NameError`) — ошибки в твоём же коде станут невидимыми;
- голый `except:` ловит вообще всё, включая `KeyboardInterrupt` (Ctrl+C) и `SystemExit`, — он запрещён;
- в `try` держать минимум кода: только строки, которые могут бросить ожидаемое исключение.

In [ ]:
def parse_float(s):
    try:
        value = float(s)
    except ValueError as e:
        print(f"  не число: {s!r} ({e})")
        return None
    else:
        print(f"  успех: {value}")
        return value
    finally:
        print("  finally выполняется всегда")

for s in ["3.14", "abc", ""]:
    print(repr(s))
    parse_float(s)

In [ ]:
# Почему широкий except опасен: опечатка в имени переменной тоже "обработана"
def risky(s):
    try:
        return flaot(s)          # опечатка: NameError
    except Exception:
        return None              # ошибка в коде молча проглочена

print(risky("1.5"))              # None — и ты не узнаешь, что функция вообще не работает

print(FileNotFoundError.__mro__)  # цепочка родительских классов исключения

### Оператор `with`

`with open(...) as f:` гарантирует, что файл будет закрыт при выходе из блока — **и при обычном выходе, и при исключении**. Объект, который умеет так работать, называется *менеджером контекста*: у него есть методы `__enter__` и `__exit__`.

Это прямой аналог идиомы **RAII** в C++, к которой мы придём в месяце 3: ресурс освобождается автоматически, когда заканчивается его область действия.

In [ ]:
try:
    with open(PATH, encoding="utf-8") as f:
        first = f.readline()
        raise RuntimeError("что-то пошло не так внутри with")
except RuntimeError as e:
    print("поймали:", e)

print("файл закрыт?", f.closed)   # True: with закрыл файл, несмотря на исключение

### Задача 1. Парсер без pandas и без модуля `csv`

`read_measurements(path)` возвращает кортеж `(records, errors)`:
- `records` — список словарей `{"sample": str, "temp": float, "value": float}` для корректных строк, в порядке следования в файле;
- `errors` — список номеров строк файла (нумерация с 1), которые не удалось разобрать;
- пустые строки, строки, начинающиеся с `#`, и заголовок `sample,temp,value` пропускаются и ошибками **не** считаются.

Строка считается ошибочной, если в ней не ровно 3 поля или `temp`/`value` не преобразуются в `float`.

**Подсказки:**
- `enumerate(f, start=1)` даёт пары (номер строки, строка);
- `line.strip()` убирает `\n` и пробелы по краям;
- `float("")` и `float("bad")` бросают `ValueError`;
- распаковка `s, t, v = parts` при неверном числе элементов тоже бросает `ValueError` — значит, одной ветки `except ValueError` хватит на оба случая.

Удобно вынести разбор одной строки в отдельную функцию `parse_line(line)` — она пригодится в задаче 2.

In [ ]:
def read_measurements(path):
    raise NotImplementedError

recs, errs = read_measurements(PATH)
assert [r["sample"] for r in recs] == ["A1", "A2", "A5"], recs
assert recs[0] == {"sample": "A1", "temp": 20.5, "value": 0.113}
assert errs == [5, 7], errs
print("OK")

## 3. Итераторы и генераторы

Цикл `for x in obj` под капотом делает так:
```python
it = iter(obj)            # получить итератор
while True:
    try:
        x = next(it)      # взять следующий элемент
    except StopIteration: # элементы кончились
        break
    ...тело цикла...
```

**Генератор** — функция с `yield`. Её вызов **не выполняет тело**, а возвращает объект-генератор. Тело выполняется по кусочкам: при каждом `next()` — до следующего `yield`, где функция «замирает» вместе со всеми локальными переменными.

Отсюда свойства:
- **ленивость**: значения вычисляются по одному, по требованию; весь результат никогда не хранится в памяти — можно обработать файл больше оперативной памяти;
- **одноразовость**: пройденный генератор пуст, второй проход ничего не выдаст;
- если генератор открывает файл через `with`, файл остаётся открытым, пока генератор не пройден до конца (или не закрыт через `.close()` / не удалён сборщиком мусора).

In [ ]:
def countdown(n):
    print("  старт генератора")
    while n > 0:
        print(f"  выдаю {n}")
        yield n
        n -= 1
    print("  генератор закончился")

g = countdown(3)
print("создали генератор, тело ещё не выполнялось:", g)
print("next ->", next(g))
print("next ->", next(g))
print("остаток:", list(g))
print("второй проход:", list(g))   # пусто: генератор одноразовый

### Задача 2. Генератор значений

`iter_values(path)` — **генератор** (функция с `yield`), лениво выдающий `value` корректных строк.

Требование: файл читается построчно внутри генератора. Вызывать `read_measurements` нельзя — она читает весь файл в список, и ленивость пропадёт. Переиспользуй `parse_line` из задачи 1.

**Объясни:** чем этот генератор отличается от функции, возвращающей список, с точки зрения памяти, и почему второй `list(g)` пуст.

In [ ]:
def iter_values(path):
    raise NotImplementedError

assert inspect.isgeneratorfunction(iter_values), "iter_values должна содержать yield"
g = iter_values(PATH)
print(type(g))
assert list(g) == [0.113, 0.127, 0.140]
assert list(g) == [], "почему второй проход пуст?"
print("OK")

## 4. Классы

- `class Series:` создаёт **объект-класс**. Вызов `Series(...)` создаёт **экземпляр** и вызывает у него `__init__`.
- `__init__` — не конструктор в смысле C++: объект к этому моменту уже создан (это делает `__new__`), `__init__` его только **инициализирует**.
- `self` — это сам экземпляр. `obj.method(x)` — то же, что `Series.method(obj, x)`. Имя `self` — соглашение, а не ключевое слово.
- Атрибуты экземпляра хранятся в словаре `obj.__dict__`.
- `__repr__` — однозначное представление для разработчика (используется в консоли, в отладке и внутри контейнеров). `__str__` — «красивое» представление для `print`; если его нет, используется `__repr__`.

**Две ловушки:**
1. **Атрибут класса** (объявлен прямо в теле класса) — один на все экземпляры. Если он изменяемый, все экземпляры делят один объект.
2. **Сохранение чужого объекта.** `self.values = values` сохраняет ссылку на список вызывающего, а не копию: если вызывающий потом изменит свой список, «изменится» и объект. Это та же модель «имя → объект» из блокнота 01.

In [ ]:
class Bad:
    items = []                 # атрибут КЛАССА: один список на всех
    def add(self, x):
        self.items.append(x)   # изменение общего списка

a, b = Bad(), Bad()
a.add(1)
print(b.items)                 # [1] — "чужой" элемент

class Good:
    def __init__(self):
        self.items = []        # атрибут ЭКЗЕМПЛЯРА: у каждого свой
    def add(self, x):
        self.items.append(x)

a, b = Good(), Good()
a.add(1)
print(b.items, a.__dict__)     # [] {'items': [1]}

### Задача 3. Класс `Series`

Класс хранит имя и список значений:
- `Series(name, values)` — сохраняет `name` и **копию** `values` в атрибуте `values` (внешнее изменение исходного списка не должно влиять на объект);
- `len(s)` — число значений (метод `__len__`);
- `repr(s)` — строка вида `Series('T', n=3)` (метод `__repr__`; подсказка: `{self.name!r}` даст имя в кавычках);
- `s.normalized()` — **новый** `Series` с тем же именем и значениями, линейно перенесёнными в [0, 1]: `(v − min) / (max − min)`. Исходный объект не меняется. Если все значения равны, бросить `ValueError` (иначе будет деление на ноль).

Здесь можно использовать встроенные `min` и `max`.

In [ ]:
class Series:
    pass

src = [2.0, 4.0, 6.0]
s = Series("T", src)
src.append(100.0)
assert s.values == [2.0, 4.0, 6.0], "Series хранит ссылку на чужой список вместо копии"
assert len(s) == 3 and repr(s) == "Series('T', n=3)", repr(s)
n = s.normalized()
assert n.values == [0.0, 0.5, 1.0] and n.name == "T"
assert s.values == [2.0, 4.0, 6.0], "normalized изменил исходный объект"
try:
    Series("x", [1, 1]).normalized()
    raise AssertionError("ожидался ValueError")
except ValueError:
    pass
print("OK")

## 5. Comprehensions (генераторы списков, словарей, множеств)

`[выражение for x in итерируемое if условие]` — компактная запись цикла, создающего список:
```python
result = []
for x in iterable:
    if condition:
        result.append(expression)
```

Аналогично: `{k: v for ...}` — словарь, `{x for ...}` — множество, `(x for ...)` — **генераторное выражение** (ленивое, как генератор из раздела 3; кортеж так не создаётся).

**Тонкости:**
- переменная цикла внутри comprehension не «утекает» наружу;
- `sum(x * x for x in data)` не создаёт промежуточный список, а `sum([x * x for x in data])` — создаёт;
- если comprehension не помещается в одну строку или в нём больше двух `for`, напиши обычный цикл: читаемость важнее.

In [ ]:
squares = [x * x for x in range(5)]
gen = (x * x for x in range(5))
print(squares, gen, sum(gen))
print([(i, j) for i in range(2) for j in range(3)])   # порядок for — как во вложенных циклах

### Задача 4. Comprehensions

Каждое значение — одним выражением:
- `lengths` — словарь `{слово: длина}` для слов из `words` длиннее 2 символов;
- `squares_even` — список квадратов чётных чисел от 0 до 20 включительно;
- `pairs` — множество пар `(i, j)`, где `0 <= i < j < 4`.

In [ ]:
words = ["alpha", "beta", "gamma", "pi", "omega"]
lengths = None
squares_even = None
pairs = None

assert lengths == {"alpha": 5, "beta": 4, "gamma": 5, "omega": 5}
assert squares_even == [0, 4, 16, 36, 64, 100, 144, 196, 256, 324, 400]
assert pairs == {(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)}
print("OK")

## 6. Задача 5. Вне блокнота: модуль, venv, pytest, отладчик

**Шаг 1. Виртуальное окружение.** venv — изолированная копия интерпретатора со своими пакетами, чтобы версии библиотек одного проекта не конфликтовали с другими.
```bash
python -m venv .venv
source .venv/bin/activate        # Windows: .venv\Scripts\activate
pip install pytest
pip freeze > requirements.txt    # зафиксировать версии
```

**Шаг 2. Модуль.** Перенеси `parse_line`, `read_measurements`, `iter_values` и `Series` в файл `lab_io.py`. В конце модуля добавь:
```python
if __name__ == "__main__":
    import sys
    records, errors = read_measurements(sys.argv[1])
    print(f"корректных строк: {len(records)}, ошибок: {errors}")
```
`__name__` равно `"__main__"`, только когда файл запущен как скрипт (`python lab_io.py data.csv`). При `import lab_io` этот блок не выполняется.

**Шаг 3. Тесты** в `test_lab_io.py`. pytest находит файлы `test_*.py` и функции `test_*`. Фикстура `tmp_path` — временная папка, своя для каждого теста:
```python
import pytest
from lab_io import read_measurements, Series

def test_skips_comments(tmp_path):
    p = tmp_path / "d.csv"
    p.write_text("# c\nsample,temp,value\nA1,1,2\n", encoding="utf-8")
    records, errors = read_measurements(p)
    assert len(records) == 1 and errors == []

def test_constant_series_raises():
    with pytest.raises(ValueError):
        Series("x", [1, 1]).normalized()
```
Добавь тесты: пустой файл; несуществующий путь (какое исключение должно вылетать и нужно ли его ловить внутри `read_measurements`?); файл только с битыми строками. Запуск: `pytest -v`.

**Шаг 4. Отладчик.** Пройди `read_measurements` по шагам: точка останова в VS Code или вызов `breakpoint()` в коде. Команды pdb: `n` (следующая строка), `s` (войти в функцию), `p выражение` (напечатать), `c` (продолжить), `q` (выход).

## Вопросы
1. Что произойдёт с файлом, если внутри `with` вылетит исключение? Как это связано с `__exit__`?
2. Почему голый `except:` запрещён, `except Exception:` опасен, а `except ValueError:` — правильно?
3. В каком порядке выполняются `try`, `except`, `else`, `finally` при успехе и при ошибке?
4. Что такое `self`? Где хранятся атрибуты объекта? Чем атрибут класса отличается от атрибута экземпляра?
5. Почему `Series` должен копировать входной список?
6. Как работает `for` под капотом (`iter`, `next`, `StopIteration`)? Почему генератор одноразовый?
7. Чем `[x for x in data]` отличается от `(x for x in data)`?
8. Зачем нужен venv и что делает `if __name__ == "__main__":`?